# Fine-tuning BERT models for medical report classification

## Example of synthetic data generation with Faker

In [ ]:
import os
import requests
import zipfile
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker
import random

## Download data

In [ ]:
# Dataset URL
url = "https://archive.ics.uci.edu/static/public/17/breast+cancer+wisconsin+diagnostic.zip"

# Destination folder where the file will be extracted
dest_folder = "../data"

# Destination zip file
dest_zip_file = os.path.join(dest_folder, "breast_cancer_wisconsin_diagnostic.zip")

# Destination parquet file
dest_parquet_file = os.path.join(dest_folder, "breast_cancer.parquet")

if not os.path.exists(dest_folder):

    # Create the folder if it does not exist
    print(f"Creating folder {dest_folder}...")
    os.makedirs(dest_folder, exist_ok=True)

    print(f"Folder {dest_folder} created successfully.")

if not os.path.exists(dest_zip_file):

    # Download and save the file
    print("Downloading dataset...")
    response = requests.get(url)
    response.raise_for_status()  # Raises an error if the download fails

    with open(dest_zip_file, "wb") as f:
        f.write(response.content)
    print(f"Download complete. Saved to {dest_zip_file}")

    # Extract the archive
    print("Extracting files...")
    with zipfile.ZipFile(dest_zip_file, 'r') as zip_ref:
        zip_ref.extractall(dest_folder)

    print(f"Files extracted to {dest_folder}")
else:
    print(f"Files already exist in {dest_folder}")

## Adjust columns

In [ ]:
# Path to the data file
data_file = os.path.join(dest_folder, "wdbc.data")

# Load the data
data = pd.read_csv(data_file, header=None, sep=",")

# Keep only the first 12 columns
data = data.iloc[:, :12]

# Define column names
column_names = [
    "id_number",
    "diagnosis",
    "radius",
    "texture",
    "perimeter",
    "area",
    "smoothness",
    "compactness",
    "concavity",
    "concave points",
    "symmetry",
    "fractal dimension"
]

# Assign column names to the DataFrame
data.columns = column_names

# Display the first rows of the DataFrame
data.head()

## Diagnosis distribution

In [ ]:
# Show value counts for the "diagnosis" column
data["diagnosis"].value_counts()

In [ ]:
# Plot attribute distribution by diagnosis
sns.pairplot(data, hue='diagnosis', vars=['radius', 'texture', 'perimeter', 'area'], palette='Set2')
plt.suptitle("Attribute Distribution by Diagnosis", y=1.02)
plt.show()

## Generate reports based on the data

In [ ]:
fake = Faker('en_US')

# Helper to translate diagnosis code to label
def translate_diagnosis(diagnosis):
    return "Benign" if diagnosis == "B" else "Malignant"

# Function to generate a synthetic medical report
def generate_report(row):
    patient = fake.name()
    size = round(row['radius'] * 2, 1)
    texture = "regular" if row['texture'] < 20 else "irregular"
    quadrant = random.choice(["upper left", "upper right", "lower left", "lower right"])
    diagnosis = translate_diagnosis(row['diagnosis'])

    return f"""
        Patient: {patient}
        Exam: Mammography
        Exam Date: {fake.date_this_year()}

        Description:
        A lesion of approximately {size} mm was observed, located in the {quadrant} quadrant, with {texture} borders.
        The exam suggests that the lesion presents {diagnosis.lower()} characteristics.

        Conclusion: {diagnosis}.
        Recommendation: {('Follow up with a new exam in 6 months' if diagnosis == 'Benign' else 'Refer for biopsy and oncological evaluation')}.
        """

# Apply to all rows in the dataset
data['report'] = data.apply(generate_report, axis=1)

## Verify generated reports

In [ ]:
# Show only the diagnosis and report columns
data[['id_number', 'diagnosis', 'report']].head()

In [ ]:
# Show the full report text for the first 3 malignant diagnoses
for report in data[data['diagnosis'] == 'M']['report'].head(3):
    print(report)
    print("=" * 80)

In [ ]:
# Show the full report text for the first 3 benign diagnoses
for report in data[data['diagnosis'] == 'B']['report'].head(3):
    print(report)
    print("=" * 80)

## Add noise

In [ ]:
# # Introduce noise into the data
# def add_noise(row):

#     # row['radius'] += random.uniform(-1, 1)  # Small random adjustment to radius
#     # row['texture'] += random.uniform(-1, 1)  # Small random adjustment to texture

#     if random.random() < 0.05:  # 5% chance of flipping the diagnosis
#         row['diagnosis'] = "B" if row['diagnosis'] == "M" else "M"

#     return row

# data = data.apply(add_noise, axis=1)

## Save data

In [ ]:
# Save the DataFrame as a parquet file in the data folder
data.to_parquet(os.path.join(dest_folder, "breast_cancer.parquet"))